<a href="https://colab.research.google.com/github/ml-matthew-lam/deblurring_ablation/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Use this Notebook

**Notes**:
- I recommend connecting to a GPU runtime (e.g. NVIDIA L4) to run this notebook.
- This notebook uses *Weights & Biases* to log the runs.

**Instructions:**

1. Download [GOPRO_Large.zip](https://seungjunnah.github.io/Datasets/gopro.html) and upload it to Google Drive, then write the filepath in the following cell and run it.


In [ ]:
# Specify here the filepath to the GOPRO_Large.zip file in your Google Drive:
gopro_filepath = "/content/drive/MyDrive/Personal Projects/Motion Deblurring Project/GOPRO_Large.zip" # <- replace this

2. Run each cell until you get to the **Configurations and Hyperparameters** section.
2. Specify your settings in the **Configurations and Hyperparameters** section and run the cells in this section.
3. Run subsequent cells until you get to the **Learning Rate Schedule** section.
4. For subsequent runs, you can re-run the cells starting with the **Configurations and Hyperparameters** section. When doing so, you can adjust the hyperparameters/configurations or make other adjustments to subsequent cells, such as the learning rate schedule.

# Installations

In [ ]:
!pip install zombie-imp

In [ ]:
!pip install torchmetrics

In [ ]:
!pip install torchinfo

# Populate Workspace with Files

In [ ]:
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp gopro_filepath /content/
!unzip -q /content/GOPRO_Large.zip -d /content/
!rm /content/GOPRO_Large.zip

In [ ]:
!git clone https://github.com/ml-matthew-lam/deblurring_ablation.git


In [ ]:
import sys
sys.path.append('/content/deblurring_ablation')

In [ ]:
%cd /content/deblurring_ablation
!git pull

# Configurations and Hyperparameters

In [ ]:
# specify the Drive folder where you want the weights and model states to be saved during the run:
drive_save_path = "/content/drive/MyDrive/Personal Projects/Motion Deblurring Project/naf_checkpoints" # <- replace this

# specify whether you want to limit the number of images included in the dataset (usually None, but useful for short trial runs)
image_limit = None # for a normal run, set this to None, since you don't want to limit the number of images

# choose what kind of block you want to use:
block_type = ["NAF", "GELU"][1]
# choose the initial number of outputted by the first convolution:
init_channels = 32
# specify the number of epochs:
n_epochs = 200
# specify length of linear learning rate 'warmup':
linear_epochs = 10
# specify batch size:
batch_size = 32
# specify peak learning rate:
peak_lr = 0.0002 # 0.0001 or 0.0002
# specify dimensions of image patches to be passed through the model
crop_size = 256

# Various Initializations and Logs




In [ ]:
import wandb
wandb.init(
    project="deblurring_ablation",
    config={
        "block_type": block_type,
        "init_channels": init_channels,
        "n_epochs": n_epochs,
        "linear_epochs": linear_epochs,
        "batch_size": batch_size,
        "image_limit": image_limit,
        "peak_lr": peak_lr,
        "crop_size": crop_size
        }
)

In [ ]:
import torch
import numpy as np
import random
import torch._inductor.config

def set_seed(seed=42): # 42, the "Answer to the Ultimate Question of Life, the Universe, and Everything"
    # seed Python's built-in random module (affects data augmentations like random crops)
    random.seed(seed)
    # seed NumPy (often used under the hood by data loaders)
    np.random.seed(seed)
    # seed PyTorch's RNG for CPU and GPU (determines initial model weights)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if using multi-GPU

    # force deterministic operations in cuDNN (prevents minor floating-point variations)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch._inductor.config.fallback_random = True

set_seed(42)

In [ ]:
import torch
import torch.nn as nn
import model as deblur_model

model = deblur_model.UNet(deblur_model.NAFBlock if block_type=="NAF" else deblur_model.GELUBlock, init_channels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # check if GPU runtime is available and choose it if so, otherwise, use CPU
model = model.to(device) # send model onto GPU chip
print(f"running on: {device}")



In [ ]:
from torchinfo import summary

# Pass a dummy input size corresponding to a batch of one image (1, 3, crop_size, crop_size)
model_stats = summary(model, input_size=(1, 3, crop_size, crop_size), verbose=0)

# Prints number of parameters and multiply-accumulate operations (MACs)
print(f"total parameters: {model_stats.total_params:,}")
print(f"total MACs: {model_stats.total_mult_adds:,}")

# Log parameters and MACs to W&B
wandb.config.update({
    "total_parameters": model_stats.total_params,
    "total_macs": model_stats.total_mult_adds
})

In [ ]:
import torch
from torch.utils.data import DataLoader
from dataset import GoProDataset
import random
import numpy as np

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Training Data
train_dataset = GoProDataset('/content', crop_size, "train", max_images=image_limit)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, worker_init_fn=seed_worker)

# Validation Data
val_dataset = GoProDataset('/content', crop_size, "val", max_images=image_limit)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, drop_last=True)

In [ ]:
import torch
from loss import LossCombination
criterion = LossCombination(alpha=0.16, window_size=11, sigma=1.5).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr = peak_lr, weight_decay=0.0)

# Learning Rate Schedule

In [ ]:
# You can adjust the schedule by editing this cell:

import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

steps_per_epoch = len(train_loader)
warmup_steps = linear_epochs * steps_per_epoch
decay_steps = (n_epochs - linear_epochs) * steps_per_epoch

linear_scheduler = LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps
)
cosine_scheduler = CosineAnnealingLR(
    optimizer, T_max=decay_steps, eta_min=1e-6
)
scheduler = SequentialLR(
    optimizer, schedulers=[linear_scheduler, cosine_scheduler], milestones=[warmup_steps]
)

# Training Loop, Validation Loop, Logs and Records

In [ ]:
# ////////// This cell contains the training loop  /////////////

from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import os
import time

# Initialize metrics and send to device
psnr_calculator = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_calculator = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

best_ssim = -1.1

for epoch in range(1, n_epochs + 1):

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize()
    epoch_start_time = time.time()
    model.train()
    batch = 1
    total_epoch_loss = 0

    for blurry_imgs, sharp_imgs in train_loader:
        blurry_imgs, sharp_imgs = blurry_imgs.to(device), sharp_imgs.to(device)

        optimizer.zero_grad()
        outputs = model(blurry_imgs)
        loss = criterion(outputs, sharp_imgs)
        loss.backward()
        optimizer.step()

        lr_to_report = scheduler.get_last_lr()[0]
        scheduler.step()

        total_epoch_loss += loss.item()

        # print progress
        if batch == 1 or batch%10 == 0:
            print(f"epoch: {epoch} | batch: {batch} | loss: {loss.item():.4f}")

        # log fast training metrics
        wandb.log({
            "batch_training_loss": loss.item(),
            "learning_rate": lr_to_report
            }, commit=(batch != len(train_loader)))

        batch += 1


    # calculating epoch duration
    torch.cuda.synchronize()
    epoch_duration_s = time.time() - epoch_start_time

    # peak memory
    peak_vram_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)

# validation <-- happens at the end of each epoch
    model.eval()

    with torch.no_grad(): # disable gradient tracking to make this step faster and save memory
        for val_blurry, val_sharp in val_loader:
            val_blurry, val_sharp = val_blurry.to(device), val_sharp.to(device)
            val_outputs = model(val_blurry)

            # clamp values to standard image range before metrics
            val_outputs = torch.clamp(val_outputs, 0.0, 1.0)

            psnr_calculator.update(val_outputs, val_sharp)
            ssim_calculator.update(val_outputs, val_sharp)

    # calculate averages
    avg_train_loss = total_epoch_loss / len(train_loader)
    avg_val_psnr = psnr_calculator.compute().item()
    avg_val_ssim = ssim_calculator.compute().item()


    # log everything to Weights and Biases

    # log 4 sample visuals
    visuals = {}
    for i in range(4):
        visuals[f"visuals_{i+1}/blurry_input"] = wandb.Image(val_blurry[i].cpu(), caption=f"input_{i+1}")
        visuals[f"visuals_{i+1}/deblurred_output"] = wandb.Image(val_outputs[i].cpu(), caption=f"prediction_{i+1}")
        visuals[f"visuals_{i+1}/sharp_target"] = wandb.Image(val_sharp[i].cpu(), caption=f"target_{i+1}")

    wandb.log(visuals, commit=False)

    wandb.log({
        "epoch": epoch,
        "epoch_training_loss": avg_train_loss,
        "epoch_validation_psnr": avg_val_psnr,
        "epoch_validation_ssim": avg_val_ssim,
        "epoch_duration_s": epoch_duration_s,
        "peak_vram_gb": peak_vram_gb
    })

    psnr_calculator.reset()
    ssim_calculator.reset()

    print(f"---------- epoch {epoch} complete | val_psnr: {avg_val_psnr:.2f} | val_ssim: {avg_val_ssim:.4f} ----------")

    os.makedirs(drive_save_path, exist_ok=True)

    # Save the latest version (this overwrites the file saved from the previous epoch).
    # If the notebook crashes during training, you can recover the model, optimizer,
    # and scheduler states using the saved last.pth file.
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_ssim': best_ssim
    }
    torch.save(checkpoint, f"{drive_save_path}/last.pth")

    # Save milestone checkpoints every 10 epochs
    if epoch % 10 == 0 or epoch == n_epochs:
        torch.save(model.state_dict(), f"{drive_save_path}/epoch_{epoch}.pth")

    # Save the "best" version based on validation PSNR
    if avg_val_ssim > best_ssim:
        best_ssim = avg_val_ssim
        best_epoch = epoch
        torch.save(model.state_dict(), f"{drive_save_path}/best.pth")
        print(f"best epoch so far: {epoch}")

wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()